# CAPT-IIoT: fixed pipeline

This is the reviewed / fixed version of the original notebook. Changes are
marked inline as `FIX 1` .. `FIX 6` and summarized here:

1. **`cap_window` no longer uses ground-truth labels** to decide which edges
   survive when a window is too big. It used to always keep every edge
   touching a malicious node and randomly thin out the rest -- for every
   window, including ones that later became the test set. That's information
   a real detector never has in advance, and it inflated every downstream
   metric. Capping is now uniform random sampling.
2. **The LSTM's per-node state is now two dense tensors instead of two Python
   dicts rebuilt with a per-node `for` loop.** No behavior change, just
   removes a large constant-factor slowdown.
3. **Contrastive pretraining now only sees windows before the train/test
   cutoff.** The cutoff is computed once, early, at the window level. After
   training, a single frozen (`no_grad`) forward pass over the *full*
   chronological sequence produces embeddings for the test-period nodes --
   exactly what deployment looks like (train once, then observe new windows).
4. **Continuous node features (uid/gid/ports/permissions) are standardized
   before going into the GNN**, fit only on nodes touched by pretrain-only
   windows. Previously these went in raw, at wildly different scales than the
   0/1 one-hot columns.
5. **`spatial_contrastive_loss`'s negative sampling is vectorized.** The
   original was a Python double loop (per edge, a `while` rejection-sampling
   loop) which was slow enough to limit how many negatives/epochs were
   practical.
6. **`NodeClassifier` is now actually defined.** It was referenced later in
   the original notebook but never defined anywhere in the saved file -- the
   notebook as given could not run top-to-bottom. Swap in your real
   architecture here if this MLP doesn't match what you used.

The single biggest efficiency bug -- `train_epoch` calling the *whole*
O(T) forward pass again after every one of the T windows' optimizer steps
(O(T^2) forward passes per epoch, which is why 43 windows took ~7
minutes/epoch) -- is folded into fix 2/3: training now does one forward
pass, accumulates all three loss terms across windows, and takes a single
optimizer step per epoch.

Things flagged in review but **not** changed in code (still worth doing):
- Walk-forward validation across multiple cutoffs, not just one 70/30 split
  (the current test slice only contains `defenceEvasion`-tactic malicious
  nodes, so it can't tell you much about generalization to other tactics).
- An auxiliary edge-level anomaly/type-prediction head.
- Embedding-space oversampling (SMOTE/ADASYN) as an alternative/complement to
  focal loss.
- Threshold tuning against a target precision (false-alarm budget) instead
  of pure F1, which matters more operationally for an IDS.

In [1]:
import pandas as pd
import numpy as np
import torch as th
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
from torch_geometric.utils import from_networkx, degree, to_undirected
from torch_geometric.data import Data
from torch_geometric.nn import MessagePassing
from sklearn.preprocessing import StandardScaler
import random

SEED = 42
random.seed(SEED); np.random.seed(SEED); th.manual_seed(SEED)
device = 'cpu'

In [2]:
def load_phase(fn, phase):
    df = pd.read_csv(fn, low_memory=False)
    df['phase'] = phase
    return df

df1 = load_phase('Phase1_Provenance.csv', 1)   # <- update paths if needed
df2 = load_phase('Phase2_Provenance.csv', 2)
df = pd.concat([df1, df2], ignore_index=True)

ENTITY_TYPES = ['Process', 'Artifact']
REL_TYPES = ['WasGeneratedBy', 'Used', 'WasTriggeredBy', 'WasDerivedFrom']

entities = df[df['type'].isin(ENTITY_TYPES)].copy()
relations = df[df['type'].isin(REL_TYPES)].copy().reset_index(drop=True)
relations['row_id'] = relations.index  # stable position for later edge-feature lookup

# entity ids repeat across Phase1/Phase2 (same continuous system) -> dedupe, OR the label
ent_label = entities.groupby('id')['label'].apply(lambda s: (s == 1).any())
entities_dedup = entities.drop_duplicates(subset='id', keep='last').set_index('id')
entities_dedup['label'] = ent_label

# propagate malicious WasDerivedFrom edges onto the artifact node they produced
# (this uses the dataset's own ground-truth edge labels to build node labels --
# that's normal supervised label construction, not the leakage issue below)
wdf_mal_to = set(relations.loc[(relations['type'] == 'WasDerivedFrom') &
                                (relations['label'] == 1), 'to'])
entities_dedup.loc[entities_dedup.index.isin(wdf_mal_to), 'label'] = True

print(f"nodes: {len(entities_dedup)}  malicious: {int(entities_dedup['label'].sum())} "
      f"({100*entities_dedup['label'].mean():.3f}%)")

nodes: 84850  malicious: 402 (0.474%)


In [3]:
TOP_K_EXE = 40

node_ids = entities_dedup.index.to_numpy()
id_to_idx = {nid: i for i, nid in enumerate(node_ids)}

top_exe = entities_dedup['exe'].value_counts().head(TOP_K_EXE).index
entities_dedup['exe_bucket'] = np.where(entities_dedup['exe'].isin(top_exe), entities_dedup['exe'], 'OTHER')
entities_dedup['exe_bucket'] = entities_dedup['exe_bucket'].fillna('NONE')
entities_dedup['subtype'] = entities_dedup['subtype'].fillna('NONE')
entities_dedup['type'] = entities_dedup['type'].fillna('NONE')
entities_dedup['protocol'] = entities_dedup['protocol'].fillna('NONE')

cat_cols = ['type', 'subtype', 'exe_bucket', 'protocol']
node_cat = pd.get_dummies(entities_dedup[cat_cols], dummy_na=False)
num_cols = ['permissions', 'uid', 'gid', 'euid', 'egid', 'remote port', 'local port']
node_num = entities_dedup[num_cols].fillna(-1)

# NOTE: node_features tensor is built later (FIX 4 cell), after the
# continuous columns have been standardized on train-only nodes. We keep the
# raw dataframe + column groups here so the later cell knows which columns
# are one-hot (never scale) vs continuous (scale).
node_feat_df = pd.concat([node_cat, node_num], axis=1).astype(float)
CAT_COLS_ENC = list(node_cat.columns)
NUM_COLS_ENC = list(node_num.columns)

node_labels = th.tensor(entities_dedup['label'].astype(int).values, dtype=th.long)

NDIM_IN = node_feat_df.shape[1]
print("node feature dim:", NDIM_IN, "| raw node_feat_df:", node_feat_df.shape)

node feature dim: 58 | raw node_feat_df: (84850, 58)


In [4]:
relations['operation'] = relations['operation'].fillna('NONE')
edge_cat = pd.get_dummies(relations[['type', 'operation']], dummy_na=False)
edge_feat_df = edge_cat.astype(float)
edge_features = th.tensor(edge_feat_df.values, dtype=th.float)
EDIM = edge_features.shape[1]

relations['src'] = relations['from'].map(id_to_idx)
relations['dst'] = relations['to'].map(id_to_idx)
valid = relations['src'].notna() & relations['dst'].notna()
print(f"edge feature dim: {EDIM} | valid edges: {valid.sum()}/{len(relations)}")
relations = relations[valid].copy()
relations['src'] = relations['src'].astype(int)
relations['dst'] = relations['dst'].astype(int)

edge feature dim: 17 | valid edges: 318796/318796


In [5]:
WINDOW_SECONDS = 14400   # 4-hour windows; shrink if a window is too slow, widen if too many windows
MAX_EDGES_PER_WINDOW = 8000

t0 = relations['time'].min()
relations['window'] = np.floor((relations['time'] - t0) / WINDOW_SECONDS).astype(np.int64)

# ---------------------------------------------------------------------------
# FIX 1: cap_window used to be label-aware -- it always kept every edge that
# touched a malicious node (mal_idx, built from ground-truth node_labels) and
# only randomly dropped the rest. That means the labels decided which local
# structure survives into the graph, for every window including the ones
# that later become the test set. A real detector never has that information
# in advance, so this silently made malicious nodes look structurally
# distinctive and inflated every downstream metric. The fix below caps with
# a label-blind rule: uniform random sampling over all edges in the window.
# ---------------------------------------------------------------------------
def cap_window(g, seed=SEED):
    if len(g) <= MAX_EDGES_PER_WINDOW:
        return g
    return g.sample(n=MAX_EDGES_PER_WINDOW, random_state=seed).sort_index()

windows = []
for w, g in relations.groupby('window'):
    g = cap_window(g)
    if len(g) > 0:
        windows.append(g)

print(f"total windows: {len(windows)}")
print(pd.Series([len(g) for g in windows]).describe())
window_phase = [int(g['phase'].mode()[0]) for g in windows]
print("phase of each window:", window_phase[:10], "...")

# ---------------------------------------------------------------------------
# FIX 3 (part 1): decide the train/test cutoff now, at the window level,
# before feature scaling or contrastive pretraining touch the data.
# Pretraining will only ever see windows[:cutoff]; windows[cutoff:] are held
# out for evaluation, same as they will be at deployment time.
# ---------------------------------------------------------------------------
phase2_window_positions = [i for i, p in enumerate(window_phase) if p == 2]
cutoff = phase2_window_positions[int(len(phase2_window_positions) * 0.7)]
print(f"phase2 windows: {len(phase2_window_positions)} | cutoff window index: {cutoff}")

train_window_node_ids = np.union1d(
    pd.concat([g['src'] for g in windows[:cutoff]]).unique(),
    pd.concat([g['dst'] for g in windows[:cutoff]]).unique(),
)
print(f"nodes touched by pretrain-only windows: {len(train_window_node_ids)}")

total windows: 43
count      43.000000
mean     6854.720930
std       975.024477
min      2434.000000
25%      6230.500000
50%      7052.000000
75%      7401.500000
max      8000.000000
dtype: float64
phase of each window: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] ...
phase2 windows: 19 | cutoff window index: 37
nodes touched by pretrain-only windows: 70121


In [6]:
# ---------------------------------------------------------------------------
# FIX 4: node_features used to go into the GNN completely unscaled (raw
# uid/gid/port integers sitting next to 0/1 one-hot columns), so message
# passing was dominated by whichever raw numeric column had the largest
# magnitude. We standardize only the continuous columns, fit on nodes
# touched by pretrain-only windows so no held-out-period information leaks
# into the scaling statistics either.
# ---------------------------------------------------------------------------
scaler_fit_rows = node_feat_df.iloc[train_window_node_ids][NUM_COLS_ENC]

num_scaler = StandardScaler()
num_scaler.fit(scaler_fit_rows.values)

node_feat_scaled = node_feat_df.copy()
node_feat_scaled[NUM_COLS_ENC] = num_scaler.transform(node_feat_df[NUM_COLS_ENC].values)

node_features = th.tensor(node_feat_scaled.values, dtype=th.float)
print("node_features (scaled):", node_features.shape)

node_features (scaled): torch.Size([84850, 58])


In [7]:
time_graphs = []

for g in windows:
    touched = np.union1d(g['src'].unique(), g['dst'].unique())
    g2l = {gl: i for i, gl in enumerate(touched)}

    local_src = g['src'].map(g2l).to_numpy()
    local_dst = g['dst'].map(g2l).to_numpy()
    edge_index = th.tensor(np.vstack([local_src, local_dst]), dtype=th.long)

    edge_attr = edge_features[g['row_id'].to_numpy()]
    edge_attr = th.nan_to_num(edge_attr, nan=0.0)

    x = node_features[touched]
    y = node_labels[touched]

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    data.h = edge_attr  # alias: original model's forward_gnn_only() reads g.h
    data.y = y
    data.global_node_ids = th.tensor(touched, dtype=th.long)
    data.phase = int(g['phase'].mode()[0])

    time_graphs.append(data)

print(f"built {len(time_graphs)} time graphs")
print("example window 0:", time_graphs[0])

# FIX 3 (part 2): pretraining only ever sees windows before the cutoff.
time_graphs_train = time_graphs[:cutoff]
print(f"pretrain-only windows: {len(time_graphs_train)} / {len(time_graphs)} total")

C:\Users\gpgup_wz3q3rl\AppData\Local\Temp\ipykernel_14604\1106217928.py:11: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  edge_attr = edge_features[g['row_id'].to_numpy()]


built 43 time graphs
example window 0: Data(x=[4147, 58], edge_index=[2, 8000], edge_attr=[8000, 17], h=[8000, 17], y=[4147], global_node_ids=[4147], phase=1)
pretrain-only windows: 37 / 43 total


In [8]:
class SAGELayer(MessagePassing):
    def __init__(self, ndim_in, edim, ndim_out, activation):
        super(SAGELayer, self).__init__(aggr='mean')
        self.W_msg = nn.Linear(ndim_in + edim, ndim_out)
        self.W_apply = nn.Linear(ndim_in + ndim_out, ndim_out)
        self.activation = activation

    def forward(self, x, edge_index, edge_attr, weight=None):
        if weight is not None:
            edge_attr = edge_attr * weight.unsqueeze(-1)
        out = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        out = self.W_msg(out)
        out = torch.cat([x, out], dim=1)
        out = self.W_apply(out)
        return self.activation(out)

    def message(self, x_j, edge_attr):
        return torch.cat([x_j, edge_attr], dim=1)


class SAGE(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim, activation, dropout):
        super(SAGE, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(SAGELayer(ndim_in, edim, 128, activation))
        self.layers.append(SAGELayer(128, edim, ndim_out, activation))
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, edge_index, edge_attr, weight=None):
        for i, layer in enumerate(self.layers):
            if i != 0:
                x = self.dropout(x)
            x = layer(x, edge_index, edge_attr, weight) if weight is not None else layer(x, edge_index, edge_attr)
        return x


class EGraphSAGE_LSTM_Model(nn.Module):
    def __init__(self, node_in_dim, edge_dim, gnn_out, lstm_hidden, dropout=0.1):
        super(EGraphSAGE_LSTM_Model, self).__init__()
        self.gnn = SAGE(node_in_dim, gnn_out, edge_dim, activation=F.relu, dropout=dropout)
        self.lstm_hidden = lstm_hidden
        self.lstm_cell = nn.LSTMCell(gnn_out, lstm_hidden)
        self.fuse = nn.Linear(node_in_dim + lstm_hidden, node_in_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, time_graphs, device, num_global_nodes):
        # -------------------------------------------------------------
        # FIX 2: per-node state used to live in two Python dicts rebuilt
        # with a `for n in nodes.tolist(): ...` loop on every window. That
        # loop -- plus the caller re-invoking this whole O(T) forward pass
        # after every window's optimizer step (see FIX 2/3 in train_epoch)
        # -- was the main reason 43 tiny windows took ~7 minutes/epoch.
        # Two dense state tensors + fancy indexing replace dict + python
        # loop with no behavior change (never-seen nodes still start from
        # a zero state).
        # -------------------------------------------------------------
        state_h = torch.zeros(num_global_nodes, self.lstm_hidden, device=device)
        state_c = torch.zeros(num_global_nodes, self.lstm_hidden, device=device)
        all_graph_embs = {}
        node_emb_all = torch.zeros(num_global_nodes, self.lstm_hidden).to(device)

        for t in range(len(time_graphs)):
            g = time_graphs[t]
            x = g.x.to(device)
            edge_index = g.edge_index.to(device)
            edge_attr = g.edge_attr.to(device)
            nodes = g.global_node_ids

            prev_h = state_h[nodes]
            prev_c = state_c[nodes]

            x_input = self.fuse(torch.cat([x, prev_h], dim=1))
            h_gnn = self.gnn(x_input, edge_index, edge_attr)
            h_time, c_time = self.lstm_cell(h_gnn, (prev_h, prev_c))

            state_h = state_h.clone()
            state_c = state_c.clone()
            state_h[nodes] = h_time
            state_c[nodes] = c_time

            node_emb_all = node_emb_all.clone()
            node_emb_all[nodes] = h_time
            all_graph_embs[t] = node_emb_all
        return all_graph_embs

    def forward_gnn_only(self, g, device):
        x = g.x.to(device)
        edge_index = g.edge_index.to(device)
        edge_attr = g.h.to(device)
        return self.gnn(x, edge_index, edge_attr)

In [9]:
def get_gca_node_weights_exact(edge_index, num_nodes=None):
    edge_index_ = to_undirected(edge_index)
    if num_nodes is None:
        num_nodes = edge_index_.max().item() + 1
    deg = degree(edge_index_[1], num_nodes=num_nodes).float()
    s = th.log(deg.clamp(min=1))
    s_max, s_mean = s.max(), s.mean()
    denom = s_max - s_mean
    return th.ones_like(s) if denom == 0 else (s_max - s) / denom

def get_gca_feature_mask_exact(z, data, global_avg_weights, base_strength=0.2, noise_scale=0.1, min_corrupt=0.01, max_corrupt=0.3):
    w = global_avg_weights.to(z.device)[data.global_node_ids]
    norm_w = (w - w.min()) / (w.max() - w.min() + 1e-6)
    corrupt = (base_strength * norm_w).clamp(min_corrupt, max_corrupt).unsqueeze(1)
    noise = torch.randn_like(z) * noise_scale
    return (1 - corrupt) * z + corrupt * noise

def get_negative_feature_mask_exact(z, data, global_avg_weights, base_strength=0.5, noise_scale=0.3, min_corrupt=0.05, max_corrupt=0.5):
    w = global_avg_weights.to(z.device)[data.global_node_ids]
    inv_w = (w.max() - w) / (w.max() - w.min() + 1e-6)
    corrupt = (base_strength * inv_w).clamp(min_corrupt, max_corrupt).unsqueeze(1)
    noise = torch.randn_like(z) * noise_scale
    return (1 - corrupt) * z + corrupt * noise

def spatial_contrastive_loss(z, edge_index, tau=0.5, num_neg=5):
    # -------------------------------------------------------------------
    # FIX 5: this used to be a Python double loop (per edge, a `while`
    # rejection-sampling loop up to num_neg*20 tries) -- slow enough to cap
    # how many negatives/epochs were practical. The vectorized version
    # below samples all negatives in one shot and only re-draws the case
    # where a negative equals its own anchor. We drop the "negative must
    # not already be a neighbor" check for speed; in a sparse graph the
    # chance of sampling a true neighbor as a negative is low, and this is
    # a standard trade-off in contrastive-learning implementations.
    # -------------------------------------------------------------------
    src, dst = edge_index
    z = F.normalize(z, dim=1)
    N = z.size(0)
    E = src.size(0)
    pos_sim = (z[src] * z[dst]).sum(dim=1, keepdim=True) / tau

    neg = torch.randint(0, N, (E, num_neg), device=z.device)
    bad = neg == src.unsqueeze(1)
    while bad.any():
        neg[bad] = torch.randint(0, N, (int(bad.sum().item()),), device=z.device)
        bad = neg == src.unsqueeze(1)

    neg_z = z[neg]                                              # [E, num_neg, dim]
    neg_sims = (z[src].unsqueeze(1) * neg_z).sum(dim=-1) / tau  # [E, num_neg]

    logits = torch.cat([pos_sim, neg_sims], dim=1)
    labels = torch.zeros(E, dtype=torch.long, device=z.device)
    return F.cross_entropy(logits, labels)

def feature_contrastive_loss(z_active, data, global_avg_weights, temperature=0.2):
    z_pos = get_gca_feature_mask_exact(z_active, data, global_avg_weights)
    z_neg = get_negative_feature_mask_exact(z_active, data, global_avg_weights)
    anchor, positive, negative = F.normalize(z_active, dim=1), F.normalize(z_pos, dim=1), F.normalize(z_neg, dim=1)
    pos_sim = (anchor * positive).sum(dim=1) / temperature
    neg_sim = (anchor * negative).sum(dim=1) / temperature
    logits = torch.stack([pos_sim, neg_sim], dim=1)
    labels = torch.zeros(logits.shape[0], dtype=torch.long, device=anchor.device)
    return F.cross_entropy(logits, labels)

class TemporalPredictorLoss(nn.Module):
    def __init__(self, feature_dim):
        super().__init__()
        self.predictor = nn.Sequential(nn.Linear(feature_dim, feature_dim), nn.BatchNorm1d(feature_dim), nn.ReLU(), nn.Linear(feature_dim, feature_dim))

    def forward(self, z_t, z_t_next, z_t_far, g_t, g_next, g_far, penalty_weight=0.05):
        z_t_pred = F.normalize(self.predictor(z_t), dim=-1)
        z_t_next_norm, z_t_far_norm = F.normalize(z_t_next, dim=-1), F.normalize(z_t_far, dim=-1)
        ids_t, ids_next, ids_far = g_t.global_node_ids.cpu().numpy(), g_next.global_node_ids.cpu().numpy(), g_far.global_node_ids.cpu().numpy()
        loss_align = torch.tensor(0.0, device=z_t.device)
        loss_uniform = torch.tensor(0.0, device=z_t.device)
        common_pos = np.intersect1d(ids_t, ids_next)
        if len(common_pos) > 0:
            idx = torch.from_numpy(common_pos).to(z_t.device)
            sim_pos = (z_t_pred[idx] * z_t_next_norm[idx]).sum(dim=-1)
            loss_align = 2 - 2 * sim_pos.mean()
        common_neg = np.intersect1d(ids_t, ids_far)
        if len(common_neg) > 0:
            idx = torch.from_numpy(common_neg).to(z_t.device)
            sim_neg = (z_t_pred[idx] * z_t_far_norm[idx]).sum(dim=-1)
            loss_uniform = F.relu(sim_neg - 0.1).mean()
        return loss_align + penalty_weight * loss_uniform

def get_grad_norms(losses, shared_layer_params):
    shared_layer_params = list(shared_layer_params)
    norms = []
    for loss in losses:
        if loss == 0.0 or (isinstance(loss, torch.Tensor) and (not loss.requires_grad or loss.item() == 0.0)):
            norms.append(0.0); continue
        grads = torch.autograd.grad(loss, shared_layer_params, retain_graph=True, allow_unused=True)
        total = sum((g.data.norm(2).item() ** 2 for g in grads if g is not None)) ** 0.5
        norms.append(total)
    return norms

def compute_grad_balanced_weights(g_a, g_b, g_c, epsilon=1e-8):
    g_a, g_b, g_c = float(g_a or 0), float(g_b or 0), float(g_c or 0)
    total = g_a + g_b + g_c
    if total <= 1e-12:
        return (0.0, 0.5, 0.5)
    return (g_a / total, g_b / total, g_c / total)

In [10]:
num_global_nodes = node_features.shape[0]
ndim_out, lstm_hidden = 128, 64
model = EGraphSAGE_LSTM_Model(NDIM_IN, EDIM, ndim_out, lstm_hidden, dropout=0.15).to(device)

global_sum = th.zeros(num_global_nodes)
global_count = th.zeros(num_global_nodes)
for g in time_graphs_train:     # FIX 3: pretrain-only windows, not the full sequence
    local_w = get_gca_node_weights_exact(g.edge_index, num_nodes=g.num_nodes)
    global_sum[g.global_node_ids] += local_w
    global_count[g.global_node_ids] += 1

final_avg_weights = global_sum / global_count.clamp(min=1)
print("final_avg_weights shape:", final_avg_weights.shape)

final_avg_weights shape: torch.Size([84850])


In [11]:
temporal_loss_module = TemporalPredictorLoss(lstm_hidden).to(device)
optimizer = torch.optim.Adam(list(model.parameters()) + list(temporal_loss_module.parameters()), lr=1e-3)

def train_epoch(model, temporal_loss_module, time_graphs, optimizer, device, num_global_nodes,
                 tau=0.3, num_neg=10, d1=1, d2=4):
    # -------------------------------------------------------------------
    # FIX 2/3: the original loop called `model(time_graphs, ...)` -- itself
    # an O(T) forward over the whole sequence -- once per window, inside a
    # loop over T windows: O(T^2) forward passes per epoch (T~37 here means
    # ~37 full sequence passes/epoch), which is why this took ~7
    # minutes/epoch for a tiny graph. We now do ONE forward pass per epoch,
    # accumulate the three loss components across all windows, estimate the
    # grad-norm balancing weights once from the accumulated sums, and take
    # a single optimizer step. This keeps the original multi-task balancing
    # idea but at epoch granularity instead of per-window, and removes the
    # pathological recompute.
    # -------------------------------------------------------------------
    model.train(); temporal_loss_module.train()
    optimizer.zero_grad()
    all_graph_embs = model(time_graphs, device, num_global_nodes)
    shared_params = list(model.fuse.parameters())

    spatial_loss_sum = torch.tensor(0.0, device=device)
    feature_loss_sum = torch.tensor(0.0, device=device)
    temporal_loss_sum = torch.tensor(0.0, device=device)

    T = len(time_graphs)
    for t in range(T):
        g = time_graphs[t]
        z_t = all_graph_embs[t]
        spatial_loss_sum = spatial_loss_sum + spatial_contrastive_loss(
            z_t[g.global_node_ids], g.edge_index, tau=tau, num_neg=num_neg)
        feature_loss_sum = feature_loss_sum + feature_contrastive_loss(
            z_t[g.global_node_ids], g, final_avg_weights)

        if t + d2 < T:
            temporal_loss_sum = temporal_loss_sum + temporal_loss_module(
                z_t, all_graph_embs[t + d1], all_graph_embs[t + d2],
                g, time_graphs[t + d1], time_graphs[t + d2])
        elif t - d2 >= 0 and t + d1 < T:
            temporal_loss_sum = temporal_loss_sum + temporal_loss_module(
                z_t, all_graph_embs[t + d1], all_graph_embs[t - d2],
                g, time_graphs[t + d1], time_graphs[t - d2])

    grad_norms = get_grad_norms([temporal_loss_sum, spatial_loss_sum, feature_loss_sum], shared_params)
    alpha, beta, gamma = compute_grad_balanced_weights(*grad_norms)
    total_loss = alpha * temporal_loss_sum + beta * spatial_loss_sum + gamma * feature_loss_sum

    total_loss.backward()
    optimizer.step()
    return total_loss.item() / T

epochs = 30   # cheap epochs now (O(T), not O(T^2)) -- room to train longer than the original 10
best_loss = float('inf')
for epoch in range(epochs):
    import time as _time
    t0 = _time.time()
    loss = train_epoch(model, temporal_loss_module, time_graphs_train, optimizer, device, num_global_nodes)
    print(f"Epoch {epoch+1}/{epochs} | loss={loss:.6f} | {_time.time()-t0:.1f}s")
    if loss < best_loss:
        best_loss = loss
        torch.save(model.state_dict(), 'capt_pretrain_weights.pth')
        torch.save(model, 'capt_pretrain_model.pth')
        print(f"  saved new best (loss={best_loss:.6f})")

Epoch 1/30 | loss=2.075091 | 11.0s
  saved new best (loss=2.075091)
Epoch 2/30 | loss=1.645366 | 9.8s
  saved new best (loss=1.645366)
Epoch 3/30 | loss=1.509879 | 9.8s
  saved new best (loss=1.509879)
Epoch 4/30 | loss=1.404144 | 10.1s
  saved new best (loss=1.404144)
Epoch 5/30 | loss=1.373806 | 10.0s
  saved new best (loss=1.373806)
Epoch 6/30 | loss=1.266800 | 10.0s
  saved new best (loss=1.266800)
Epoch 7/30 | loss=1.231860 | 9.9s
  saved new best (loss=1.231860)
Epoch 8/30 | loss=1.302844 | 9.8s
Epoch 9/30 | loss=1.433911 | 10.1s
Epoch 10/30 | loss=1.459295 | 9.9s
Epoch 11/30 | loss=1.415962 | 10.0s
Epoch 12/30 | loss=1.486430 | 9.9s
Epoch 13/30 | loss=1.564588 | 10.0s
Epoch 14/30 | loss=1.614910 | 9.8s
Epoch 15/30 | loss=1.641568 | 9.9s
Epoch 16/30 | loss=1.631085 | 10.0s
Epoch 17/30 | loss=1.558296 | 10.1s
Epoch 18/30 | loss=1.451944 | 10.0s
Epoch 19/30 | loss=1.268867 | 9.9s
Epoch 20/30 | loss=1.127024 | 10.0s
  saved new best (loss=1.127024)
Epoch 21/30 | loss=1.018191 | 10.0

In [12]:
model.load_state_dict(torch.load('capt_pretrain_weights.pth', map_location=device))
model.eval()

# ---------------------------------------------------------------------------
# FIX 3 (part 3): the encoder was trained only on windows[:cutoff]. We now
# run ONE frozen inference pass over the FULL chronological sequence
# (train + test windows) to obtain embeddings for test-period nodes too.
# No gradient is computed and no parameter update happens here -- this is
# exactly what deployment looks like: a trained model observing new windows
# as they arrive.
# ---------------------------------------------------------------------------
with torch.no_grad():
    all_graph_embs = model(time_graphs, device, num_global_nodes)
    final_embeddings = all_graph_embs[len(time_graphs) - 1].clone()  # latest known state per node

print("final_embeddings:", final_embeddings.shape)

final_embeddings: torch.Size([84850, 64])


In [13]:
last_seen_window = th.full((num_global_nodes,), -1, dtype=th.long)
for t, g in enumerate(time_graphs):
    last_seen_window[g.global_node_ids] = t

# `cutoff` here is the SAME window index computed earlier (FIX 3, part 1) --
# reused for consistency instead of being recomputed from time_graphs.
touched_mask = last_seen_window >= 0
train_mask = touched_mask & (last_seen_window < cutoff)
test_mask = touched_mask & (last_seen_window >= cutoff)

train_idx = th.where(train_mask)[0]
test_idx = th.where(test_mask)[0]

y_all = node_labels
print(f"train nodes: {len(train_idx)}  malicious: {int(y_all[train_idx].sum())}")
print(f"test nodes:  {len(test_idx)}  malicious: {int(y_all[test_idx].sum())}")
print(f"untouched (excluded, no graph context): {(~touched_mask).sum().item()}")

train nodes: 65662  malicious: 318
test nodes:  10681  malicious: 46
untouched (excluded, no graph context): 8507


In [14]:
# ---------------------------------------------------------------------------
# FIX 6: `NodeClassifier` was referenced later in the original notebook but
# never defined anywhere in the saved file -- it must have only existed in a
# live kernel session, so the notebook as given could not run top-to-bottom.
# A small MLP head is added here; swap in your real architecture if this
# doesn't match what you actually used.
# ---------------------------------------------------------------------------
class NodeClassifier(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 2),
        )

    def forward(self, x):
        return self.net(x)


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

clf_features_raw = th.cat([final_embeddings, node_features], dim=1)
CLF_IN_DIM = clf_features_raw.shape[1]

tr_idx_np = train_idx.numpy()
y_tr_np = y_all[train_idx].numpy()
tr_sub, val_sub = train_test_split(tr_idx_np, test_size=0.15, stratify=y_tr_np, random_state=SEED)
tr_sub = th.tensor(tr_sub, dtype=th.long)
val_sub = th.tensor(val_sub, dtype=th.long)

clf_scaler = StandardScaler()
clf_scaler.fit(clf_features_raw[tr_sub].numpy())
clf_features = th.tensor(clf_scaler.transform(clf_features_raw.numpy()), dtype=th.float)

print(f"train/val: {len(tr_sub)}/{len(val_sub)}  |  val malicious: {int(y_all[val_sub].sum())}")
print("feature stats after scaling (should be ~0 mean, ~1 std on train):",
      clf_features[tr_sub].mean().item(), clf_features[tr_sub].std().item())

train/val: 55812/9850  |  val malicious: 48
feature stats after scaling (should be ~0 mean, ~1 std on train): 2.92220203590432e-09 0.9958932995796204


In [15]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        logp = F.log_softmax(logits, dim=1)
        p = logp.exp()
        logp_t = logp.gather(1, targets.unsqueeze(1)).squeeze(1)
        p_t = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        loss = -((1 - p_t) ** self.gamma) * logp_t
        if self.alpha is not None:
            loss = loss * self.alpha[targets]
        return loss.mean()

clf = NodeClassifier(CLF_IN_DIM).to(device)

n_pos = int(y_all[tr_sub].sum())
n_neg = len(tr_sub) - n_pos
alpha = th.tensor([1.0 / n_neg, 1.0 / n_pos])
alpha = alpha / alpha.sum() * 2
loss_fn = FocalLoss(alpha=alpha, gamma=2.0)
opt = torch.optim.Adam(clf.parameters(), lr=1e-3, weight_decay=1e-5)

from sklearn.metrics import f1_score, precision_recall_curve

def best_threshold(y_true, probs):
    prec, rec, thr = precision_recall_curve(y_true, probs)
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    best_i = np.nanargmax(f1s[:-1]) if len(thr) > 0 else 0
    return (thr[best_i] if len(thr) > 0 else 0.5), f1s[best_i] if len(thr) > 0 else 0.0

best_val_f1, best_state, best_thr = -1, None, 0.5
for epoch in range(150):
    clf.train()
    opt.zero_grad()
    logits = clf(clf_features[tr_sub])
    loss = loss_fn(logits, y_all[tr_sub])
    loss.backward()
    opt.step()

    clf.eval()
    with torch.no_grad():
        val_probs = F.softmax(clf(clf_features[val_sub]), dim=1)[:, 1].numpy()
    thr, f1 = best_threshold(y_all[val_sub].numpy(), val_probs)

    if f1 > best_val_f1:
        best_val_f1, best_thr = f1, thr
        best_state = {k: v.clone() for k, v in clf.state_dict().items()}

    if epoch % 10 == 0:
        print(f"epoch {epoch:3d} | loss={loss.item():.4f} | val_F1(best_thr)={f1:.4f} | thr={thr:.3f}")

clf.load_state_dict(best_state)
print(f"\nbest val F1: {best_val_f1:.4f}  at threshold: {best_thr:.3f}")

epoch   0 | loss=0.0043 | val_F1(best_thr)=0.0266 | thr=0.484
epoch  10 | loss=0.0026 | val_F1(best_thr)=0.4071 | thr=0.541
epoch  20 | loss=0.0021 | val_F1(best_thr)=0.4074 | thr=0.645
epoch  30 | loss=0.0018 | val_F1(best_thr)=0.4694 | thr=0.737
epoch  40 | loss=0.0017 | val_F1(best_thr)=0.4842 | thr=0.817
epoch  50 | loss=0.0016 | val_F1(best_thr)=0.4783 | thr=0.876
epoch  60 | loss=0.0016 | val_F1(best_thr)=0.4646 | thr=0.847
epoch  70 | loss=0.0015 | val_F1(best_thr)=0.4646 | thr=0.845
epoch  80 | loss=0.0013 | val_F1(best_thr)=0.4646 | thr=0.827
epoch  90 | loss=0.0012 | val_F1(best_thr)=0.5303 | thr=0.637
epoch 100 | loss=0.0011 | val_F1(best_thr)=0.5983 | thr=0.706
epoch 110 | loss=0.0011 | val_F1(best_thr)=0.6140 | thr=0.783
epoch 120 | loss=0.0011 | val_F1(best_thr)=0.6140 | thr=0.790
epoch 130 | loss=0.0011 | val_F1(best_thr)=0.6306 | thr=0.779
epoch 140 | loss=0.0011 | val_F1(best_thr)=0.6306 | thr=0.810

best val F1: 0.6542  at threshold: 0.842


In [16]:
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

clf.eval()
with torch.no_grad():
    test_probs = F.softmax(clf(clf_features[test_idx]), dim=1)[:, 1].numpy()
test_pred = (test_probs >= best_thr).astype(int)

y_test = y_all[test_idx].numpy()
print(classification_report(y_test, test_pred, target_names=['benign', 'malicious'], zero_division=0))
print("PR-AUC:", average_precision_score(y_test, test_probs))
print("Confusion matrix:\n", confusion_matrix(y_test, test_pred))

ent_sublabel = entities.groupby('id')['subLabel'].apply(
    lambda s: next((v for v in s if pd.notna(v) and v != 0 and v != '0'), 0)
)
node_sublabel = entities_dedup.index.to_series().map(ent_sublabel).fillna(0)
wdf_mal_sub = relations[(relations['type'] == 'WasDerivedFrom') & (relations['label'] == 1)].set_index('to')['subLabel']
node_sublabel.update(wdf_mal_sub)
node_sublabel_arr = node_sublabel.reindex(entities_dedup.index).values

sub_test = node_sublabel_arr[test_idx.numpy()]
breakdown = pd.DataFrame({'subLabel': sub_test, 'true': y_test, 'pred': test_pred})
breakdown = breakdown[breakdown['true'] == 1]
print(breakdown.groupby('subLabel').apply(lambda d: pd.Series({
    'n': len(d), 'caught': (d['pred'] == 1).sum(), 'recall': (d['pred'] == 1).mean()
})))

              precision    recall  f1-score   support

      benign       1.00      1.00      1.00     10635
   malicious       0.81      0.65      0.72        46

    accuracy                           1.00     10681
   macro avg       0.90      0.83      0.86     10681
weighted avg       1.00      1.00      1.00     10681

PR-AUC: 0.591215801067854
Confusion matrix:
 [[10628     7]
 [   16    30]]
                   n  caught    recall
subLabel                              
defenceEvasion  46.0    30.0  0.652174
